<a href="https://colab.research.google.com/github/25061839/gemini-rag-study-assistant/blob/main/Gemini_RAG_Study_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U google-genai

In [ ]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="用简单的中文解释什么是人工智能。"
)

print(response.text)

In [ ]:
!pip install -q pymupdf

In [ ]:
import fitz

pdf_path = "/content/006-Queue.pdf"

doc = fitz.open(pdf_path)

print("PDF 页数：", len(doc))

In [ ]:
import pymupdf

pdf_path = "/content/006-Queue.pdf"

doc = pymupdf.open(pdf_path)

all_text = ""

for page_num, page in enumerate(doc):
    text = page.get_text()
    all_text += f"\n\n===== Page {page_num + 1} =====\n\n"
    all_text += text

print("PDF 页数：", len(doc))
print("提取的文字长度：", len(all_text))

In [ ]:
print(all_text[:5000])

In [ ]:
def split_text(text, chunk_size=1000, overlap=200):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end]
        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
chunks = split_text(
    all_text,
    chunk_size=1000,
    overlap=200
)

print("Chunk 数量：", len(chunks))

In [ ]:
for i, chunk in enumerate(chunks):
    print(f"\n========== Chunk {i+1} ==========")
    print(chunk[:500])

In [ ]:
for i, page in enumerate(doc):
    text = page.get_text().strip()
    print(f"Page {i+1}: {len(text)} characters")

In [ ]:
!pip install -q chromadb

In [ ]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("Gemini Client 创建成功")

In [ ]:
test_text = chunks[0]

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=test_text
)

print("Embedding 生成成功")

In [ ]:
embedding = result.embeddings[0].values

print("Embedding 长度：", len(embedding))
print("前10个数字：", embedding[:10])

In [ ]:
import chromadb

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="wia1002_queue_notes"
)

print("ChromaDB collection 创建成功")

In [ ]:
embeddings = []

for i, chunk in enumerate(chunks):
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=chunk
    )

    embedding = result.embeddings[0].values
    embeddings.append(embedding)

    print(f"Chunk {i+1}/{len(chunks)} embedding 完成")

In [ ]:
print("Chunk 数量：", len(chunks))
print("Embedding 数量：", len(embeddings))
print("第一个 Embedding 维度：", len(embeddings[0]))

In [ ]:
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=embeddings
)

print("所有 Chunks 已成功存入 ChromaDB")

In [ ]:
question = "What are the two important operations for a Queue?"

In [ ]:
question_result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=question
)

question_embedding = question_result.embeddings[0].values

print("Question embedding 生成成功")
print("维度：", len(question_embedding))

In [ ]:
results = collection.query(
    query_embeddings=[question_embedding],
    n_results=3
)

print("找到的相关 Chunks：")
print(results["documents"])

In [ ]:
retrieved_chunks = results["documents"][0]

context = "\n\n".join(retrieved_chunks)

prompt = f"""
You are a university study assistant.

Answer the student's question using ONLY the provided course material.

Course material:
{context}

Student question:
{question}

If the answer cannot be found in the course material, say:
"I cannot find the answer in the provided course material."

Give a clear and concise answer.
"""

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)

print(response.text)

In [ ]:
print("Documents:", collection.count())

test_results = collection.query(
    query_embeddings=[
        client.models.embed_content(
            model="gemini-embedding-001",
            contents="What is FIFO?"
        ).embeddings[0].values
    ],
    n_results=3
)

print("Documents:")
print(test_results["documents"])

print("\nMetadatas:")
print(test_results["metadatas"])

In [ ]:
def rag_answer(question, n_results=3):

    # 1. Convert question into embedding
    question_result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=question
    )

    question_embedding = question_result.embeddings[0].values

    # 2. Search ChromaDB
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )

    # 3. Get retrieved documents
    documents = results["documents"][0]

    # 4. Build context
    context_parts = []

    for i, document in enumerate(documents):

        context_parts.append(
            f"[Retrieved Chunk {i+1}]\n{document}"
        )

    context = "\n\n".join(context_parts)

    # 5. Create prompt
    prompt = f"""
You are a university study assistant for WIA1002 Data Structure.

Answer the student's question using ONLY the provided course material.

Course material:
{context}

Student question:
{question}

Instructions:
- Give a clear and accurate answer.
- Explain the concept in simple language.
- Do not invent information.
- If the answer cannot be found in the course material, say:
"I cannot find the answer in the provided course material."

Answer:
"""

    # 6. Ask Gemini
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    # 7. Extract page numbers directly from document text
    import re

    pages = []

    for document in documents:

        found_pages = re.findall(
            r"===== Page (\d+) =====",
            document
        )

        pages.extend(found_pages)

    pages = sorted(set(
        int(page) for page in pages
    ))

    # 8. Return answer + sources
    return response.text, pages

In [ ]:
answer, pages = rag_answer("What is FIFO?")

print("Answer:")
print(answer)

print("\n📚 Sources:")
print("Pages:", pages)

In [ ]:
import chromadb

collection_v2 = chroma_client.get_or_create_collection(
    name="wia1002_page_chunks"
)

print("New collection created.")

In [ ]:
print([x for x in globals().keys() if "page" in x.lower()])

In [ ]:
print(type(pages))
print("数量：", len(pages))

print("\n第一个元素：")
print(pages[0])

In [ ]:
for name, value in list(globals().items()):
    if isinstance(value, list) and len(value) > 0:
        first = value[0]

        if isinstance(first, str) and len(first) > 100:
            print(
                "变量名:", name,
                "| 数量:", len(value),
                "| 第一个元素长度:", len(first)
            )

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.lower().endswith(".pdf"):
            print(os.path.join(root, file))

In [ ]:
import fitz

pdf_path = "/content/006-Queue.pdf"

doc = fitz.open(pdf_path)

page_texts = []

for page_num in range(len(doc)):
    text = doc[page_num].get_text()

    page_texts.append(text)

print("PDF 页数：", len(page_texts))
print("第1页文字长度：", len(page_texts[0]))
print("第8页文字：")
print(page_texts[7][:500])

In [ ]:
import chromadb

collection_v2 = chroma_client.get_or_create_collection(
    name="wia1002_queue_pages"
)

print("Collection 创建成功")

In [ ]:
print("目前数量：", collection_v2.count())

In [ ]:
page_chunks = []
page_metadatas = []
page_ids = []

for i, text in enumerate(page_texts):

    page_number = i + 1

    # 如果某一页没有文字，也保存下来
    if not text.strip():
        text = "[This page contains no extractable text.]"

    page_chunks.append(text)

    page_metadatas.append({
        "page": page_number
    })

    page_ids.append(
        f"wia1002_page_{page_number}"
    )

print("Chunks 数量：", len(page_chunks))
print("第一个 ID：", page_ids[0])
print("第一个 metadata：", page_metadatas[0])

In [ ]:
embeddings = []

batch_size = 10

for i in range(0, len(page_chunks), batch_size):

    batch = page_chunks[i:i + batch_size]

    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=batch
    )

    batch_embeddings = [
        item.values
        for item in result.embeddings
    ]

    embeddings.extend(batch_embeddings)

    print(
        f"完成 {min(i + batch_size, len(page_chunks))}/{len(page_chunks)}"
    )

print("Embedding 数量：", len(embeddings))
print("Embedding 长度：", len(embeddings[0]))

In [ ]:
collection_v2.add(
    ids=page_ids,
    documents=page_chunks,
    metadatas=page_metadatas,
    embeddings=embeddings
)

print("所有页面成功存入 ChromaDB！")
print("Documents:", collection_v2.count())

In [ ]:
check = collection_v2.get(
    limit=5
)

print(check["ids"])
print(check["metadatas"])

In [ ]:
def rag_answer(question, n_results=3):

    # 1. Question embedding
    question_result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=question
    )

    question_embedding = question_result.embeddings[0].values

    # 2. Search the new collection
    results = collection_v2.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    # 3. Build context
    context_parts = []

    for document, metadata in zip(documents, metadatas):

        page = metadata["page"]

        context_parts.append(
            f"[Page {page}]\n{document}"
        )

    context = "\n\n".join(context_parts)

    # 4. Prompt
    prompt = f"""
You are a university study assistant for WIA1002 Data Structure.

Answer the student's question using ONLY the provided course material.

Course material:
{context}

Student question:
{question}

Instructions:
- Give a clear and accurate answer.
- Explain the concept in simple language.
- Do not invent information.
- If the answer cannot be found in the course material, say:
"I cannot find the answer in the provided course material."

Answer:
"""

    # 5. Generate answer
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    # 6. Source pages
    pages = sorted(set(
        metadata["page"]
        for metadata in metadatas
    ))

    return response.text, pages

In [ ]:
answer, pages = rag_answer("What is FIFO?")

print("Answer:")
print(answer)

print("\n📚 Sources:")
print("Pages:", pages)

In [ ]:
import time

questions = [
    "What is FIFO?",
    "What are the two important operations in a queue?",
    "What is the difference between a regular queue and a priority queue?",
    "Which data structure is best for implementing a queue?"
]

for question in questions:

    print("=" * 60)
    print("Question:", question)

    try:
        answer, pages = rag_answer(question)

        print("\nAnswer:")
        print(answer)

        print("\n📚 Sources:")
        print("Pages:", pages)

    except Exception as e:
        print("\n❌ Error:")
        print(e)

    print()

    # Wait before sending the next request
    time.sleep(5)

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

def ask_ai(question):

    # 防止用户没有输入问题
    if not question or not question.strip():
        return "Please enter a question.", ""

    try:
        # 调用我们之前已经完成的 RAG
        answer, pages = rag_answer(
            question.strip(),
            n_results=3
        )

        # Format source pages
        if pages:
            sources = "📚 Course Sources: " + ", ".join(
                [f"Page {p}" for p in pages]
            )
        else:
            sources = "📚 Course Sources: No page information available."

        return answer, sources

    except Exception as e:
        return f"Something went wrong:\n{str(e)}", ""


with gr.Blocks(title="WIA1002 AI Study Assistant") as demo:

    gr.Markdown(
        """
        # 🎓 WIA1002 AI Study Assistant

        Ask questions about **Data Structure** using your course material.

        The AI will search the course material first and then generate an answer.
        """
    )

    question = gr.Textbox(
        label="Your Question",
        placeholder="Example: What is FIFO?",
        lines=3
    )

    ask_button = gr.Button("🤖 Ask AI")

    answer = gr.Textbox(
        label="AI Answer",
        lines=8
    )

    sources = gr.Textbox(
        label="📚 Sources",
        lines=2
    )

    ask_button.click(
        fn=ask_ai,
        inputs=question,
        outputs=[answer, sources]
    )

    gr.Markdown(
        """
        ---
        **Course:** WIA1002 Data Structure
        **Knowledge Base:** 006-Queue.pdf
        **Powered by:** Gemini + ChromaDB + RAG
        """
    )


demo.launch(share=True)

In [ ]:
import gradio as gr

def chatbot(question):
    if not question.strip():
        return "请输入你的问题。"

    try:
        answer = rag_answer(question)

        # 如果 rag_answer 返回的是 tuple
        if isinstance(answer, tuple):
            answer = answer[0]

        return answer

    except Exception as e:
        return f"发生错误：{e}"


demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(
        label="请输入你的课程问题",
        placeholder="例如：What is FIFO?"
    ),
    outputs=gr.Textbox(
        label="AI 学习助手回答"
    ),
    title="Gemini + RAG 学习助手",
    description="基于课程 PDF 的智能问答系统"
)

demo.launch()

In [ ]:
import gradio as gr

def upload_pdf(pdf):
    if pdf is None:
        return "❌ Please upload a PDF file."

    filename = pdf.split("/")[-1]

    return f"✅ PDF uploaded successfully: {filename}"


with gr.Blocks(title="AI Course Tutor") as demo2:

    gr.Markdown(
        """
        # 🎓 AI Course Tutor

        ### RAG-powered University Course Assistant
        """
    )

    pdf_file = gr.File(
        label="📄 Upload Course PDF",
        file_types=[".pdf"],
        type="filepath"
    )

    upload_button = gr.Button("📚 Upload PDF")

    status = gr.Textbox(
        label="System Status"
    )

    upload_button.click(
        fn=upload_pdf,
        inputs=pdf_file,
        outputs=status
    )

    gr.Markdown("---")

    question = gr.Textbox(
        label="💬 Your Question",
        placeholder="Example: What is FIFO?",
        lines=3
    )

    ask_button = gr.Button("🤖 Ask AI")

    answer = gr.Textbox(
        label="🧠 AI Answer",
        lines=8
    )

    sources = gr.Textbox(
        label="📚 Sources",
        lines=2
    )


demo2.launch(share=True)

In [ ]:
!pip install -q pymupdf chromadb

In [ ]:
import fitz
import chromadb

print("✅ PyMuPDF imported successfully")
print("✅ ChromaDB imported successfully")

In [ ]:
from google import genai
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")

print("API Key loaded:", API_KEY is not None)

In [ ]:
client = genai.Client(api_key=API_KEY)

print("✅ Gemini client created")

In [ ]:
import chromadb

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="course_documents"
)

print("✅ ChromaDB collection created")

In [ ]:
print("Gemini client:", client is not None)
print("ChromaDB collection:", collection is not None)

In [ ]:
def build_knowledge_base(pdf_path):

    if pdf_path is None:
        return "❌ Please upload a PDF first."

    # 清空旧知识库
    try:
        existing_data = collection.get()

        if existing_data["ids"]:
            collection.delete(ids=existing_data["ids"])

    except Exception as e:
        print("Clear database message:", e)

    # 打开 PDF
    doc = fitz.open(pdf_path)

    documents = []
    metadatas = []
    ids = []

    chunk_id = 0

    # 逐页读取 PDF
    for page_number, page in enumerate(doc, start=1):

        text = page.get_text().strip()

        # 如果这一页没有文字，就跳过
        if not text:
            continue

        # 每个 chunk 最多 1000 个字符
        chunk_size = 1000

        for start in range(0, len(text), chunk_size):

            chunk = text[start:start + chunk_size].strip()

            if not chunk:
                continue

            documents.append(chunk)

            # 保存来源页码
            metadatas.append({
                "page": page_number
            })

            ids.append(f"chunk_{chunk_id}")

            chunk_id += 1

    total_pages = len(doc)
    doc.close()

    # 检查是否成功读取文字
    if len(documents) == 0:
        return "❌ No readable text found in this PDF."

    # 显示处理信息
    print(f"PDF pages: {total_pages}")
    print(f"Total chunks: {len(documents)}")

    # ==========================
    # Generate Embeddings
    # ==========================

    embeddings = []

    for i, text in enumerate(documents):

        result = client.models.embed_content(
            model="gemini-embedding-001",
            contents=text
        )

        embedding = result.embeddings[0].values

        embeddings.append(embedding)

        print(f"Embedding {i + 1}/{len(documents)} completed")

    # ==========================
    # Save to ChromaDB
    # ==========================

    collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas
    )

    return (
        "✅ Knowledge base built successfully!\n\n"
        f"📄 Total Pages: {total_pages}\n"
        f"📦 Total Chunks: {len(documents)}\n"
        f"🧠 Embeddings: {len(embeddings)}\n"
        "💾 Saved to ChromaDB"
    )

In [ ]:
import gradio as gr

with gr.Blocks(title="AI Course Tutor") as demo:

    gr.Markdown("""
    # 🎓 AI Course Tutor

    ### RAG-powered University Course Assistant

    Upload your course PDF, build a knowledge base,
    and ask questions about the material.
    """)

    pdf_file = gr.File(
        label="📄 Upload Course PDF",
        file_types=[".pdf"],
        type="filepath"
    )

    build_button = gr.Button("📚 Build Knowledge Base")

    status = gr.Textbox(
        label="System Status",
        lines=5
    )

    build_button.click(
        fn=build_knowledge_base,
        inputs=pdf_file,
        outputs=status
    )

    gr.Markdown("---")

    question = gr.Textbox(
        label="💬 Your Question",
        placeholder="Example: What is FIFO?",
        lines=3
    )

    ask_button = gr.Button("🤖 Ask AI")

    answer = gr.Textbox(
        label="🧠 AI Answer",
        lines=8
    )

    sources = gr.Textbox(
        label="📚 Sources",
        lines=2
    )

demo.launch(share=True)

In [ ]:
print("Documents:", collection.count())

In [ ]:
test = collection.get()

print("Number of documents:", len(test["documents"]))
print("First page:", test["metadatas"][0]["page"])

In [ ]:
def rag_answer(question, n_results=3):

    # ==========================
    # 1. Convert question to embedding
    # ==========================

    question_result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=question
    )

    question_embedding = question_result.embeddings[0].values

    # ==========================
    # 2. Search ChromaDB
    # ==========================

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )

    retrieved_documents = results["documents"][0]
    retrieved_metadatas = results["metadatas"][0]

    # ==========================
    # 3. Get source pages
    # ==========================

    pages = []

    for metadata in retrieved_metadatas:

        if metadata and "page" in metadata:
            pages.append(metadata["page"])

    # Remove duplicate pages
    pages = sorted(list(set(pages)))

    # ==========================
    # 4. Build context
    # ==========================

    context = ""

    for i, document in enumerate(retrieved_documents):

        page = retrieved_metadatas[i]["page"]

        context += f"\n\n--- Page {page} ---\n"
        context += document

    # ==========================
    # 5. Build prompt
    # ==========================

    prompt = f"""
You are an AI study assistant for a university Data Structure course.

Answer the student's question using ONLY the course material provided below.

If the answer cannot be found in the course material,
say that the information is not available in the provided material.

Keep the answer clear and suitable for a university student.

Course material:
{context}

Student question:
{question}
"""

    # ==========================
    # 6. Generate answer
    # ==========================

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text, pages

In [ ]:
answer, pages = rag_answer("What is FIFO?")

print("ANSWER:")
print(answer)

print("\n📚 SOURCE PAGES:")
print(pages)

In [ ]:
 import gradio as gr

def ask_ai(question):

    if question is None or not question.strip():
        return "Please enter a question.", ""

    try:

        answer, pages = rag_answer(
            question.strip(),
            n_results=3
        )

        if pages:
            source_text = "📚 Sources: " + ", ".join(
                [f"Page {page}" for page in pages]
            )
        else:
            source_text = "📚 Sources: Not available"

        return answer, source_text

    except Exception as e:

        return f"❌ Error: {str(e)}", ""


with gr.Blocks(title="AI Course Tutor") as demo:

    gr.Markdown("""
    # 🎓 AI Course Tutor

    ### RAG-powered University Course Assistant

    Upload your course PDF and ask questions based on the material.
    """)

    # ==========================
    # PDF Upload
    # ==========================

    pdf_file = gr.File(
        label="📄 Upload Course PDF",
        file_types=[".pdf"],
        type="filepath"
    )

    build_button = gr.Button(
        "📚 Build Knowledge Base"
    )

    status = gr.Textbox(
        label="System Status",
        lines=4
    )

    build_button.click(
        fn=build_knowledge_base,
        inputs=pdf_file,
        outputs=status
    )

    gr.Markdown("---")

    # ==========================
    # Question
    # ==========================

    question = gr.Textbox(
        label="💬 Your Question",
        placeholder="Example: What is FIFO?",
        lines=3
    )

    ask_button = gr.Button(
        "🤖 Ask AI"
    )

    # ==========================
    # Answer
    # ==========================

    answer = gr.Textbox(
        label="🧠 AI Answer",
        lines=8
    )

    sources = gr.Textbox(
        label="📚 Course Sources",
        lines=2
    )

    ask_button.click(
        fn=ask_ai,
        inputs=question,
        outputs=[answer, sources]
    )


demo.launch(share=True)

In [ ]:
import time

def generate_quiz():

    if collection.count() == 0:
        return "❌ Please build the knowledge base first."

    data = collection.get()

    documents = data["documents"]
    metadatas = data["metadatas"]

    context = ""

    for i in range(min(10, len(documents))):

        page = metadatas[i]["page"]

        context += f"\n--- Page {page} ---\n"
        context += documents[i]

    prompt = f"""
You are a university Data Structure teaching assistant.

Create 5 multiple-choice questions based ONLY on
the course material below.

For each question provide:

Question:
A.
B.
C.
D.

Correct Answer:
Explanation:
Source Page:

Make the questions suitable for a university beginner.

Course material:
{context}
"""

    # Retry up to 3 times for temporary 503 errors
    for attempt in range(3):

        try:

            response = client.models.generate_content(
                model="gemini-3.6-flash",
                contents=prompt
            )

            return response.text

        except Exception as e:

            print(f"Attempt {attempt + 1}/3 failed.")
            print(e)

            if attempt < 2:
                print("Waiting 10 seconds...")
                time.sleep(10)

            else:
                return (
                    "❌ Gemini is temporarily unavailable.\n"
                    "Please try again later."
                )

In [ ]:
quiz = generate_quiz()

print(quiz)

In [ ]:
test_questions = [
    {
        "question": "What is FIFO?",
        "expected_pages": [8, 9]
    },
    {
        "question": "What are the two main operations of a queue?",
        "expected_pages": [11, 13]
    },
    {
        "question": "How is a queue implemented using LinkedList?",
        "expected_pages": [14, 15, 16, 17]
    },
    {
        "question": "What is a priority queue?",
        "expected_pages": [22, 23]
    },
    {
        "question": "What is the difference between a regular queue and a priority queue?",
        "expected_pages": [22, 23, 25]
    }
]

print("Total test questions:", len(test_questions))

In [ ]:
results = []

for item in test_questions:

    question = item["question"]
    expected_pages = item["expected_pages"]

    try:
        # 1. Convert question into embedding
        result = client.models.embed_content(
            model="gemini-embedding-001",
            contents=question
        )

        question_embedding = result.embeddings[0].values

        # 2. Search ChromaDB
        search_results = collection.query(
            query_embeddings=[question_embedding],
            n_results=3
        )

        retrieved_metadatas = search_results["metadatas"][0]

        retrieved_pages = []

        for metadata in retrieved_metadatas:
            if metadata and "page" in metadata:
                retrieved_pages.append(metadata["page"])

        # Remove duplicates
        retrieved_pages = sorted(list(set(retrieved_pages)))

        # 3. Check whether correct page was retrieved
        correct = any(
            page in expected_pages
            for page in retrieved_pages
        )

        results.append({
            "question": question,
            "expected_pages": expected_pages,
            "retrieved_pages": retrieved_pages,
            "correct": correct
        })

        print("Question:", question)
        print("Expected Pages:", expected_pages)
        print("Retrieved Pages:", retrieved_pages)

        if correct:
            print("Result: ✅ Correct")
        else:
            print("Result: ❌ Incorrect")

        print("-" * 50)

    except Exception as e:
        print("❌ Error:", e)

In [ ]:
correct_count = sum(
    1 for result in results
    if result["correct"]
)

total_count = len(results)

accuracy = correct_count / total_count * 100

print("📊 Evaluation Results")
print("=" * 40)
print("Total Questions:", total_count)
print("Correct Retrievals:", correct_count)
print(f"Retrieval Accuracy: {accuracy:.2f}%")

In [ ]:
import pandas as pd

evaluation_df = pd.DataFrame(results)

evaluation_df

In [ ]:
evaluation_df.to_csv(
    "evaluation_results.csv",
    index=False
)

print("✅ Evaluation results saved!")

In [ ]:
from google.colab import files

files.download("evaluation_results.csv")

In [ ]:
%%writefile rag.py

import pymupdf
import chromadb
from google import genai


class RAGSystem:

    def __init__(self, api_key):
        # Gemini client
        self.client = genai.Client(api_key=api_key)

        # ChromaDB
        self.chroma_client = chromadb.Client()

        self.collection = self.chroma_client.get_or_create_collection(
            name="course_documents"
        )


    def build_knowledge_base(self, pdf_path):

        # Clear old data
        try:
            existing_data = self.collection.get()

            if existing_data["ids"]:
                self.collection.delete(
                    ids=existing_data["ids"]
                )

        except Exception:
            pass


        # Open PDF
        doc = pymupdf.open(pdf_path)

        documents = []
        metadatas = []
        ids = []

        chunk_id = 0


        # Extract and split text
        for page_number, page in enumerate(doc, start=1):

            text = page.get_text().strip()

            if not text:
                continue

            chunk_size = 1000

            for start in range(0, len(text), chunk_size):

                chunk = text[
                    start:start + chunk_size
                ].strip()

                if not chunk:
                    continue

                documents.append(chunk)

                metadatas.append({
                    "page": page_number
                })

                ids.append(
                    f"chunk_{chunk_id}"
                )

                chunk_id += 1


        total_pages = len(doc)

        doc.close()


        if len(documents) == 0:
            return "No readable text found."


        # Generate embeddings
        embeddings = []

        for text in documents:

            result = self.client.models.embed_content(
                model="gemini-embedding-001",
                contents=text
            )

            embedding = result.embeddings[0].values

            embeddings.append(embedding)


        # Store in ChromaDB
        self.collection.add(
            ids=ids,
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas
        )


        return (
            "Knowledge base built successfully!\n"
            f"Total Pages: {total_pages}\n"
            f"Total Chunks: {len(documents)}\n"
            f"Embeddings: {len(embeddings)}"
        )


    def ask(self, question, n_results=3):

        # Convert question to embedding
        result = self.client.models.embed_content(
            model="gemini-embedding-001",
            contents=question
        )

        question_embedding = result.embeddings[0].values


        # Search ChromaDB
        results = self.collection.query(
            query_embeddings=[question_embedding],
            n_results=n_results
        )


        retrieved_documents = results["documents"][0]

        retrieved_metadatas = results["metadatas"][0]


        # Collect source pages
        pages = []

        for metadata in retrieved_metadatas:

            if metadata and "page" in metadata:

                pages.append(
                    metadata["page"]
                )


        pages = sorted(
            list(set(pages))
        )


        # Build context
        context = ""

        for i, document in enumerate(
            retrieved_documents
        ):

            page = retrieved_metadatas[i]["page"]

            context += (
                f"\n\n--- Page {page} ---\n"
            )

            context += document


        # RAG prompt
        prompt = f"""
You are an AI study assistant for a university
Data Structure course.

Answer the student's question using ONLY
the course material provided below.

If the answer cannot be found in the course
material, say that the information is not
available in the provided material.

Keep the answer clear and suitable for
a university student.

Course material:
{context}

Student question:
{question}
"""


        # Generate answer
        response = self.client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt
        )


        return response.text, pages

In [ ]:
!ls -l rag.py

In [ ]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

print("API key loaded:", bool(os.environ.get("GEMINI_API_KEY")))

In [ ]:
!find . -maxdepth 2 -type f | sort

In [ ]:
%%writefile app.py

import os
import gradio as gr

from rag import RAGSystem


# =========================
# Get Gemini API Key
# =========================

API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError(
        "GEMINI_API_KEY is not set. "
        "Please set your Gemini API key first."
    )


# =========================
# Initialize RAG system
# =========================

rag = RAGSystem(API_KEY)


# =========================
# Build Knowledge Base
# =========================

def build_knowledge_base(pdf_path):

    if pdf_path is None:
        return "❌ Please upload a PDF first."

    try:
        result = rag.build_knowledge_base(pdf_path)

        return "✅ " + result

    except Exception as e:
        return f"❌ Error: {str(e)}"


# =========================
# Ask AI
# =========================

def ask_ai(question):

    if not question or not question.strip():
        return "Please enter a question.", ""

    try:

        answer, pages = rag.ask(
            question.strip(),
            n_results=3
        )

        if pages:

            sources = (
                "📚 Sources: "
                + ", ".join(
                    [f"Page {p}" for p in pages]
                )
            )

        else:

            sources = "📚 Sources: Not available"

        return answer, sources

    except Exception as e:

        return f"❌ Error: {str(e)}", ""


# =========================
# Gradio Interface
# =========================

with gr.Blocks(
    title="AI Course Study Assistant"
) as demo:

    gr.Markdown(
        """
        # 🎓 AI Course Study Assistant

        Upload your university course PDF and
        ask questions using an AI-powered
        Retrieval-Augmented Generation (RAG) system.
        """
    )


    # PDF Upload

    gr.Markdown("## 📚 1. Upload Course Material")

    pdf_file = gr.File(
        label="Upload PDF",
        file_types=[".pdf"],
        type="filepath"
    )

    build_button = gr.Button(
        "📖 Build Knowledge Base"
    )

    status = gr.Textbox(
        label="Knowledge Base Status",
        lines=4
    )

    build_button.click(
        fn=build_knowledge_base,
        inputs=pdf_file,
        outputs=status
    )


    # Question Answering

    gr.Markdown("## 🤖 2. Ask the AI")

    question = gr.Textbox(
        label="Your Question",
        placeholder="e.g. What is FIFO?"
    )

    ask_button = gr.Button(
        "🔍 Ask AI"
    )

    answer = gr.Textbox(
        label="AI Answer",
        lines=8
    )

    sources = gr.Textbox(
        label="Source Pages",
        lines=2
    )

    ask_button.click(
        fn=ask_ai,
        inputs=question,
        outputs=[
            answer,
            sources
        ]
    )


# =========================
# Launch
# =========================

demo.launch(
    share=True
)

In [ ]:
!ls -l app.py

In [ ]:
%%writefile requirements.txt

google-genai
chromadb
pymupdf
gradio
pandas

In [ ]:
!cat requirements.txt

In [ ]:
%%writefile .gitignore

__pycache__/
*.pyc
.env
.env.*
*.ipynb_checkpoints
.DS_Store

In [ ]:
!cat .gitignore

In [ ]:
!python app.py

In [ ]:
%%writefile README.md